# Chapter 2 · The self-healing agent loop

In [Chapter 1](./01_otel_rag_pipeline.ipynb) we built the OTel grounding pipeline as a function. Here we drop it into an autonomous **ReAct** loop (Think → Act → Observe → Reflect) — the same shape as the north-star [`otel.py`](../otel.py) agent — so the agent grounds its diagnosis in **real OTel semantic retrieval** instead of a keyword match.

Runs on **CPU, no Databricks account needed**.

> OTel = **Open Telco**, not OpenTelemetry.

In [ ]:
# Databricks (serverless / standard cluster): RUN THIS FIRST — installs deps, then restarts Python.
# (Only the Databricks ML runtime pre-ships torch + transformers; serverless does not.)
# Local Jupyter: instead run  pip install -r ../requirements.txt  in a terminal, and skip this cell.
%pip install -q "sentence-transformers>=3.0.0"
try:
    dbutils.library.restartPython()
except NameError:
    pass

## 1. The OTel grounding tool (from Chapter 1)

Embed the standards corpus once, then retrieve by meaning with an abstain gate. This becomes the agent's `retrieve_standards` tool.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("farbodtavakkoli/OTel-Embedding-335M")

CORPUS = [
    {"cite": "3GPP TS 38.214 5.2", "text": "Low SINR/CQI forces a lower MCS, reducing per-UE throughput even at moderate PRB. Low throughput with LOW PRB load indicates a radio-quality problem, not congestion."},
    {"cite": "3GPP TS 36.213 7.2", "text": "LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB above 90% across neighboring cells in the busy hour is the signature of CONGESTION."},
    {"cite": "O-RAN WG1 UC",      "text": "Congestion remediation order: load-balance to under-utilized neighbors, enable carrier aggregation, add carrier/spectrum, then cell split or new site."},
    {"cite": "3GPP TS 36.331 8.1", "text": "PCI collision between neighbor cells corrupts measurement reports and handovers, degrading SINR. Resolve PCI conflicts before RF optimization."},
    {"cite": "RF Ops Playbook",    "text": "Correlate recurring EXTERNAL_INTERFERENCE_UL alarms with low-SINR cells before adjusting antenna tilt or transmit power."},
    {"cite": "TM Forum Open API",  "text": "Standardized management interfaces enable vendor-agnostic data collection across Huawei/Ericsson/Nokia OSS/BSS for closed-loop automation."},
]
texts = [d["text"] for d in CORPUS]
index = embedder.encode(texts, normalize_embeddings=True)
ABSTAIN_BELOW = 0.35

def retrieve_standards(query, k=2):
    """OTel semantic retrieval — the agent's grounding tool."""
    q = embedder.encode(query, normalize_embeddings=True)
    scores = index @ q
    order = np.argsort(scores)[::-1][:k]
    top = [{"cite": CORPUS[i]["cite"], "text": texts[i], "score": float(scores[i])} for i in order]
    if not top or top[0]["score"] < ABSTAIN_BELOW:
        return {"grounded": False, "refs": []}
    return {"grounded": True, "refs": top}

print("grounding tool ready")

## 2. A tiny synthetic network — the agent's other tools

Stand-ins for the OSS (KPIs, alarms) and BSS (business impact) the agent queries. One scenario: low 5G throughput in a neighborhood.

In [ ]:
SCENARIO = {
    "incident": "Subscribers report slow 5G data in the Riverside neighborhood; several NR cells show low downlink throughput.",
    "region": "Riverside",
    "cells": [
        {"id": "gNB-4471_cell0", "sinr": 18.2, "prb": 41, "dl": 210, "alarm": None},
        {"id": "gNB-4471_cell1", "sinr": 4.1,  "prb": 44, "dl": 38,  "alarm": "EXTERNAL_INTERFERENCE_UL"},
        {"id": "gNB-4472_cell0", "sinr": 3.4,  "prb": 39, "dl": 33,  "alarm": "EXTERNAL_INTERFERENCE_UL"},
    ],
    "bss": {"affected_subscribers": 5400, "complaints_24h": 63, "arpu": 41.0},
}

def get_network_kpis():
    cells = SCENARIO["cells"]
    return {"count": len(cells), "avg_prb": round(sum(c["prb"] for c in cells)/len(cells), 1),
            "min_dl": min(c["dl"] for c in cells), "cells": cells}

def get_alarms():
    return {c["id"]: c["alarm"] for c in SCENARIO["cells"] if c["alarm"]}

def get_bss_data():
    b = dict(SCENARIO["bss"])
    b["revenue_at_risk_per_day"] = round(b["affected_subscribers"] * b["arpu"] / 30.0, 0)
    return b

print("OSS/BSS tools ready")

## 3. The ReAct loop

A compact, deterministic policy plans the next sensible step; each round prints **THINK → ACT → OBSERVE → REFLECT**. The grounding step calls the **OTel** tool. It ends in a **human-gated** recommendation — the agent recommends, a person approves.

In [ ]:
def run_agent():
    kb = {}
    def step(n, think, act, observe, reflect):
        print(f"\nROUND {n}")
        print(f"  THINK    {think}")
        print(f"  ACT      {act}")
        print(f"  OBSERVE  {observe}")
        print(f"  REFLECT  {reflect}")

    # R1 — pull KPIs
    kb["kpi"] = get_network_kpis()
    sick = [c for c in kb["kpi"]["cells"] if c["dl"] < 60]
    step(1, "Start with live KPIs for the affected region.", "get_network_kpis()",
         f"{kb['kpi']['count']} cells, avg PRB {kb['kpi']['avg_prb']}%, min DL {kb['kpi']['min_dl']} Mbps",
         f"{len(sick)} cells show low throughput at LOW load — smells like radio quality, not congestion.")

    # R2 — alarms
    kb["alarms"] = get_alarms()
    step(2, "Check alarms on the degraded cells.", "get_alarms()",
         ", ".join(f"{k}: {v}" for k, v in kb["alarms"].items()) or "none",
         "Recurring uplink-interference alarms line up with the low-SINR cells.")

    # R3 — ground in standards via OTel retrieval
    gq = "low downlink throughput at low PRB load with poor SINR and uplink interference"
    kb["std"] = retrieve_standards(gq)
    refs = ", ".join(f"{r['cite']} ({r['score']:.2f})" for r in kb["std"]["refs"]) if kb["std"]["grounded"] else "ABSTAINED"
    step(3, "Ground the hypothesis in telecom standards (OTel semantic retrieval).",
         f"retrieve_standards('{gq[:40]}...')", f"top refs: {refs}",
         "Standards confirm: low throughput + low load = radio quality." if kb["std"]["grounded"] else "Insufficient grounding — would ask for more data.")

    # R4 — business impact
    kb["bss"] = get_bss_data()
    step(4, "Quantify business impact before recommending action.", "get_bss_data()",
         f"{kb['bss']['affected_subscribers']:,} subs, {kb['bss']['complaints_24h']} complaints/24h, ${kb['bss']['revenue_at_risk_per_day']:,.0f}/day at risk",
         "Enough evidence to conclude.")

    # Conclude — human-gated
    cites = [r["cite"] for r in kb["std"]["refs"]] if kb["std"]["grounded"] else []
    print("\n" + "="*70)
    print("DIAGNOSIS  Radio quality — uplink interference (not congestion)")
    print(f"GROUNDED   {', '.join(cites)}")
    print(f"IMPACT     {kb['bss']['affected_subscribers']:,} subscribers, ${kb['bss']['revenue_at_risk_per_day']:,.0f}/day at risk")
    print("ACTIONS    1) mitigate the uplink interference source  2) RF-optimize tilt/power if SINR stays low")
    print("GATE       \U0001F6E1  RECOMMEND ONLY — a network engineer approves before any change goes live")
    print("="*70)

run_agent()

## 4. The swap that matters — keyword vs. OTel

Why replace the keyword match? A paraphrased incident that shares almost no words with the standard still needs to find it. Keyword overlap misses; OTel retrieves by **meaning**.

In [ ]:
def keyword_retrieve(query, k=1):
    qw = set(query.lower().split())
    scored = sorted(CORPUS, key=lambda d: len(qw & set(d["text"].lower().split())), reverse=True)
    return scored[0]["cite"], len(qw & set(scored[0]["text"].lower().split()))

paraphrase = "the cell isn't busy but data is crawling and the signal looks noisy"
kw_cite, overlap = keyword_retrieve(paraphrase)
otel = retrieve_standards(paraphrase, k=1)["refs"][0]

print(f"QUERY (paraphrased): {paraphrase}\n")
print(f"  keyword match -> {kw_cite}   (only {overlap} shared words — brittle)")
print(f"  OTel  match   -> {otel['cite']}   (semantic score {otel['score']:.2f})")
print("\nThe right standard is 3GPP TS 38.214 (low throughput at low load = radio quality).")

## What you just built

✅ An autonomous **ReAct** loop that grounds its reasoning in **real OTel semantic retrieval**, quantifies impact, and stops at a **human-gated** recommendation.  
✅ The one swap that makes the north-star loop real: OTel replaces the brittle keyword `retrieve_standards`.

The full [`otel.py`](../otel.py) wraps this in a browser UI with OpenTelemetry-style span tracing and a pluggable reasoning brain (mock → Claude → an OTel LLM).

**Chapters 3–4** take this exact pipeline to Databricks — governed serving + Vector Search ([Chapter 3](../chapters/03-productionize/README.md)), then full inference capture, monitoring, and a cost-economics ledger ([Chapter 4](../chapters/04-govern-capture/README.md)).